# aggragate country-level results to global results for each hazard

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
def get_aggregated_rates_global(hazard_type, disease_type, file_path_res, delT_all, scenario):
    res = pd.DataFrame(index=range(len(df_list)),columns=["FID", "ISO"] + [item for i in delT_all for item in (f"t{i}", f"sig{i}_95", f"sig{i}_99")])
    res.iloc[:,0:2] = df_list

    for Country in fid_info['Country']:
        file_path = os.path.join(file_path_res,f'ECA_result_{Country}_{hazard_type}.csv')
        if os.path.isfile(file_path):
            rates = pd.read_csv(file_path)
        else: 
            continue
        idx = fid_info.loc[fid_info['Country']==Country,"FID"].iloc[0].astype(int)

        if idx < 0: continue
        
        rates["signal_95"] = np.where(
            rates["precursor_rate"] == 0,
            0,
            (rates["precursor_rate"] >= rates["precursor_95"]).astype(int)
        )
        rates["signal_99"] = np.where(
            rates["precursor_rate"] == 0,
            0,
            (rates["precursor_rate"] >= rates["precursor_99"]).astype(int)
        )
        for _, row in rates.iterrows():
            t = int(row["delT"])
            res.loc[idx, f"t{t}"] = row["precursor_rate"]
            res.loc[idx, f"sig{t}_95"] = row["signal_95"]
            res.loc[idx, f"sig{t}_99"] = row["signal_99"]
    if scenario == 'all':
        out_dir = os.path.join(path, 'output', 'global', f'{scenario}')
        os.makedirs(out_dir, exist_ok=True)
        res.to_csv(os.path.join(out_dir, f'res_aggregated_rates_global_{hazard_type}_{disease_type}.csv'),index=False)
    else:
        out_dir = os.path.join(path, 'output', 'global', 'single_multi')
        os.makedirs(out_dir, exist_ok=True)
        res.to_csv(os.path.join(out_dir, f'res_aggregated_rates_global_{hazard_type}_{disease_type}.csv'),index=False) 
    return res

In [3]:
#%% set up path
path = r'/gpfs/work4/0/FWC2/Spatial_dependence/Disease_outbreak/global'
df_list = pd.read_csv(os.path.join(path,'data/shapefile/geogunit_0_list.csv'))
fid_info = pd.read_csv(os.path.join(path,'data','global_country_iso.csv'))

In [4]:
# All hazards
#  hazard_type_all = ['Flood', 'Drought', 'Storm', 'Landslide','Volcanic activity', 'Wildfire', 'Extreme temperature', 'Earthquake']
hazard_type_all = ['Natural Hazard']
# hazard_type_all = ['Mass movement']
disease_type = 'waterborne'
scenario = 'all' 
for hazard_type in hazard_type_all:
    file_path_res =  os.path.join(path, 'output', f'{disease_type}', f'{scenario}', 'country', f'{hazard_type}')
    delT_all = np.arange(13)

    out_dir = os.path.join(path, 'output', f'{disease_type}', 'country', f'{hazard_type}')
    res = get_aggregated_rates_global(hazard_type, disease_type, file_path_res, delT_all, scenario)

In [4]:
# Single/multi hazard scenarios
scenarios = ['single', 'multi']
disease_type = 'waterborne'
for scenario in scenarios:
    file_path_res =  os.path.join(path, 'output', f'{disease_type}', f'{scenario}')
    delT_all = np.arange(13)
    res = get_aggregated_rates_global(hazard_type=scenario, disease_type=disease_type, file_path_res=file_path_res, delT_all=delT_all, scenario = scenario)